In [8]:
import pandas as pd
import gc

In [4]:
df2 = pd.read_pickle('../data/df2.pkl')

In [5]:
df2.head()

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,RIDE_DIST_KM_CNT,...,DEST_STATION_NAME,DEST_MRK_ID_NUM,DEST_LATITUDE,DEST_LONGITUDE,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type
0,NaN,9200002624150,402,2025-02-22,2025-02-22 12:37:05,2025-02-22,2025-02-22 12:37:29,403,0.0,1.4,...,Woodlands TEL,402.0,1.436058,103.787939,TRAIN,Woodlands South,403.0,1.427396,103.793264,TRAIN
1,187.0,240000373709,2753,2025-02-22,2025-02-22 10:05:07,2025-02-22,2025-02-22 10:09:32,2767,0.0,1.1,...,Opp Blk 628,2753.0,1.351755,103.750199,BUS,Blk 315,2767.0,1.360038,103.746242,BUS
2,168.0,9200002624150,4344,2025-02-22,2025-02-22 12:46:47,2025-02-22,2025-02-22 13:15:51,5060,0.0,13.3,...,Bef Seletar Camp G,4344.0,1.402222,103.871388,BUS,Woodlands Int B10,5060.0,1.437037,103.786132,BUS
3,NaN,240000373709,17,2025-02-22,2025-02-22 10:40:23,2025-02-22,2025-02-22 10:41:11,28,0.0,12.4,...,Redhill,17.0,1.289635,103.816741,TRAIN,Bukit Batok,28.0,1.349033,103.749567,TRAIN
4,NaN,9190001883020,41,2025-02-22,2025-02-22 10:30:17,2025-02-22,2025-02-22 10:39:53,42,0.0,2.4,...,Tampines NSEW,41.0,1.353302,103.945145,TRAIN,Pasir Ris,42.0,1.373043,103.949285,TRAIN


In [ ]:
df2.info() # (49333714, 25)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49333714 entries, 0 to 49333713
Data columns (total 25 columns):
 #   Column              Dtype         
---  ------              -----         
 0   BUS_SVC_NUM         float64       
 1   CRD_NUM             int64         
 2   DEST_LOC_ID_NUM     int64         
 3   ENTRY_DT            datetime64[ns]
 4   ENTRY_TM            datetime64[ns]
 5   EXIT_DT             datetime64[ns]
 6   EXIT_TM             datetime64[ns]
 7   ORIG_LOC_ID_NUM     int64         
 8   RIDE_DISC_AMT       float64       
 9   RIDE_DIST_KM_CNT    float64       
 10  RIDE_FARE_AMT       float64       
 11  RIDE_ID_NUM         int64         
 12  RIDE_MIN_CNT        float64       
 13  PATRON_CATG_ID_NUM  int64         
 14  TRNSPT_MODE_CD      int64         
 15  DEST_STATION_NAME   object        
 16  DEST_MRK_ID_NUM     float64       
 17  DEST_LATITUDE       float64       
 18  DEST_LONGITUDE      float64       
 19  DEST_Travel_Type    object        
 20  

In [7]:
# select representative date
df2 = df2[df2['ENTRY_DT'] == '2025-02-11']

In [ ]:
gc.collect() # free up storage immediately to delete overwritten files - the big df2 at the start
print(df2.shape) # (3846784, 25) for 11 feb

(3846784, 25)


# Rule Based Classification

#### 1. Binary Criteria
- Drop rows with missing critical data
- The journey stage has no subsequent recorded trips on the same day and is therefore treated as a journey termination. It is the last ride of the day for the commuter.
    - Implementation: Sort by CRD_NUM and ENTRY_TM, flag rows where the next row belongs to a different CRD_NUM.
- 2 consecutive journey stages on the same bus service or train station indicates a return trip or intermediate activity. This should be classified as a new journey.
    - Implementation: Shift BUS_SVC_NUM and ORIG_LOC_ID_NUM down by 1 row to get the next stage's values, then flag rows where all 3 conditions are true simultaneously: same commuter, same bus service, and the alighting stop of stage 1 equals the boarding stop of stage 2.


In [19]:
# Drop rows with missing critical data
critical_cols = [
    'ENTRY_TM', 'EXIT_TM',
    'ORIG_LOC_ID_NUM', 'DEST_LOC_ID_NUM',
    'ORIG_LATITUDE', 'ORIG_LONGITUDE',
    'DEST_LATITUDE', 'DEST_LONGITUDE'
]

before = len(df2)
df2 = df2.dropna(subset=critical_cols).reset_index(drop=True)
after = len(df2)

print(f"Rows dropped: {before - after:,}")
print(f"Rows remaining: {after:,}")

Rows dropped: 0
Rows remaining: 3,817,059


In [ ]:
# Binary criteria
# Sort first
df2 = df2.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)

# Shift to get next row's CRD_NUM
df2['next_CRD_NUM'] = df2['CRD_NUM'].shift(-1)

# Last stage = next row belongs to a different commuter
df2['is_last_stage'] = df2['CRD_NUM'] != df2['next_CRD_NUM'] # This tells us the last ride for the specific commuter

print(df2['is_last_stage'].value_counts())
print(f"\nUnique commuters: {df2['CRD_NUM'].nunique():,}") # Check that no. if True in is_last_stage == No. of unique commuters

In [22]:
# Shift needed columns
df2['next_ORIG_LOC_ID_NUM'] = df2['ORIG_LOC_ID_NUM'].shift(-1)
df2['next_BUS_SVC_NUM']     = df2['BUS_SVC_NUM'].shift(-1)

# Flag same service return
df2['same_service_return'] = (
    (df2['CRD_NUM'] == df2['next_CRD_NUM']) &
    (df2['BUS_SVC_NUM'] == df2['next_BUS_SVC_NUM']) &
    (df2['DEST_LOC_ID_NUM'] == df2['next_ORIG_LOC_ID_NUM'])
)

print(df2['same_service_return'].value_counts())

same_service_return
False    3767169
True       49890
Name: count, dtype: int64


#### 2. Temporal criteria: 
- The binary criteria gives us pairs of consecutive stages that could be transfers. 
- Temporal criteria finds the time gap between alighting from stage 1 and boarding stage 2
- Implementation: 
    - gap = ENTRY_TM of stage 2 - EXIT_TM of stage 1. 
    - Shift ENTRY_TM and TRNSPT_MODE_CD down by 1 row to get the next stage's values. Compute the time gap between current EXIT_TM and next ENTRY_TM. Assign threshold (15 or 45 mins) based on transfer type, then flag if gap exceeds it.
    - Flag if gap > allowance.
- Allowance is given by **LTA's current window**:
    - Bus↔Bus or Bus↔Train transfers: 45 min maximum gap
    - Train↔Train transfers: 15 min maximum gap
    - Same bus service number: not a valid transfer (already handled in binary)
    - Max 5 transfers per journey
    - Overall journey duration (first to last boarding): max 2 hours


# PAUSE HERE BEFORE RUNNING BELOW. 

### Is this the correct temporal criteria?

* Note this is how TRNSPRT_MODE_CD is mapped.
1    BUS
2    RTS
3    BUS-RTS (interchange)

In [23]:
# Shift needed columns 
df2['next_ENTRY_TM']       = df2['ENTRY_TM'].shift(-1)
df2['next_TRNSPT_MODE_CD'] = df2['TRNSPT_MODE_CD'].shift(-1)

#  Compute time gap
df2['time_gap_mins'] = (
    df2['next_ENTRY_TM'] - df2['EXIT_TM']
).dt.total_seconds() / 60

# Assign threshold by transfer type 
# RTS-RTS (2-2) → 15 mins, everything else → 45 mins
df2['time_threshold'] = df2.apply(lambda row:
    15 if (row['TRNSPT_MODE_CD'] == 2 and row['next_TRNSPT_MODE_CD'] == 2)
    else 45, axis=1
)

# Flag if gap exceeds threshold
df2['exceeds_time_allowance'] = (
    (df2['CRD_NUM'] == df2['next_CRD_NUM']) &
    (df2['time_gap_mins'] > df2['time_threshold'])
)

# Flag if overall journey exceeds 2 hours 
df2['journey_start_TM'] = df2.groupby('CRD_NUM')['ENTRY_TM'].transform('min')
df2['journey_duration_mins'] = (
    df2['ENTRY_TM'] - df2['journey_start_TM']
).dt.total_seconds() / 60
df2['exceeds_2hr'] = df2['journey_duration_mins'] > 120

# Flag if transfer count exceeds 5
df2['transfer_count'] = df2.groupby('CRD_NUM').cumcount()
df2['exceeds_5_transfers'] = df2['transfer_count'] > 5

# Combined temporal flag 
df2['temporal_new_journey'] = (
    df2['exceeds_time_allowance'] |
    df2['exceeds_2hr'] |
    df2['exceeds_5_transfers']
)

print(df2['exceeds_time_allowance'].value_counts())
print(df2['exceeds_2hr'].value_counts())
print(df2['exceeds_5_transfers'].value_counts())
print(f"\nTotal temporal new journey flags: {df2['temporal_new_journey'].sum():,}")

exceeds_time_allowance
False    2537548
True     1279511
Name: count, dtype: int64
exceeds_2hr
False    2044593
True     1772466
Name: count, dtype: int64
exceeds_5_transfers
False    3692508
True      124551
Name: count, dtype: int64

Total temporal new journey flags: 2,742,629


#### 3. Spacial Criteria:
